# 08 Spatial Epidemiology — Exercises

Practice spatial analysis and visualization using the Legionnaires' disease data from Songbai Nursing Home.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## Question 1: Spatial Distribution of Case Fatality Rates

1. Compute the case fatality rate for each floor × wing (CFR = deaths / infected × 100)
2. Draw a CFR heatmap with `sns.heatmap()`
3. Which wing has the highest CFR? Is the wing with the highest CFR also the one with the highest attack rate?

In [ ]:
# TODO: compute floor × wing CFR
# TODO: sns.heatmap()
# TODO: interpret

## Question 2: Spatial Distribution of Shower Use

In Ch05 we found that shower use (`shower_use`) is a risk factor for infection.

1. Compute the proportion of shower users in each floor × wing
2. Show it with a heatmap
3. Are the wings with high shower usage also the ones with high attack rates?
4. Does this observation support the hypothesis that "the water system is the transmission route"?

In [ ]:
# TODO: compute the floor × wing shower-use proportion
# TODO: heatmap
# TODO: compare side by side with the attack rate heatmap

## Question 3 (Challenge): High-Risk Room List

You need to submit a "high-risk room list" to the infection control team:

1. Compute the attack rate for each room
2. Filter for rooms with an attack rate ≥ 75%
3. Produce a table containing: `room`, `total`, `infected`, `attack_rate`, `floor`, `wing`
4. Sort in descending order of attack rate
5. Are the high-risk rooms concentrated in a particular wing?

In [ ]:
# TODO: compute the attack rate for each room
# TODO: filter for >= 75%
# TODO: organize the table and sort it
# TODO: which wings have the most high-risk rooms?

## Question 4: Dengue Incidence Rate by District (Dengue Scenario)

A county is experiencing a dengue outbreak. The health department has case and population data for each district.

1. Compute the case count and incidence rate per 100,000 population for each district
2. Identify the district with the highest incidence rate (not necessarily the one with the most cases)
3. Plot a bar chart of incidence rate by district
4. Interpret: why compare risk across districts using incidence rate rather than raw case counts?

In [ ]:
# Dengue: case counts by district (Annan District, with more standing water, has higher risk)
rng = np.random.default_rng(841)
districts = ["安南區", "三民區", "北屯區", "板橋區", "中西區"]
pop = {"安南區": 190000, "三民區": 340000, "北屯區": 280000, "板橋區": 550000, "中西區": 78000}
rate_per_100k = {"安南區": 22, "三民區": 8, "北屯區": 5, "板橋區": 4, "中西區": 9}
_recs = []
for d in districts:
    n = rng.poisson(rate_per_100k[d] * pop[d] / 100000)
    for _ in range(n):
        _recs.append({"district": d, "age": int(rng.integers(5, 85)),
                      "serotype": rng.choice(["DENV-1", "DENV-2", "DENV-3"])})
dengue = pd.DataFrame(_recs)
region_pop = pd.DataFrame({"district": districts, "population": [pop[d] for d in districts]})
print(f"Dengue notifications: {len(dengue)} cases across {dengue['district'].nunique()} districts")

# TODO: use groupby to count dengue cases per district
# TODO: merge with region_pop and compute incidence rate per 100,000 = cases / population * 100000
# TODO: find the district with the highest "incidence rate" (hint: not necessarily the one with the most cases)
# TODO: plot a bar chart (df.plot.bar or sns.barplot) of incidence rate by district, sorted
# TODO: interpret -- why compare district risk using incidence rate rather than raw case counts?

## Question 5: COVID-19 Regional Attack Rate Hotspot Map (COVID-19 Scenario)

The city is divided into a 4×5 grid, with population and case counts available for each cell.

1. Compute the attack rate (%) for each region
2. Arrange the values into a row×col grid with `pivot`, and draw a hotspot map with `sns.heatmap`
3. Identify the region with the highest attack rate, and describe which side the hotspot is concentrated on

In [ ]:
# COVID-19: population and cases for a 4x5 grid of regions (rows A/B in the north have higher risk)
rng = np.random.default_rng(852)
_recs = []
for r in list("ABCD"):
    for c in range(1, 6):
        popn = int(rng.integers(2000, 6000))
        base = 0.03 + (0.05 if r in ("A", "B") else 0.0) + rng.normal(0, 0.008)
        cases = rng.binomial(popn, max(0.005, base))
        _recs.append({"region": f"{r}{c}", "row": r, "col": c, "population": popn, "cases": cases})
covid = pd.DataFrame(_recs)
print(f"COVID-19: {len(covid)} regions, total population {covid['population'].sum():,}, total cases {covid['cases'].sum()}")

# TODO: compute the attack rate (%) for each region = cases / population * 100
# TODO: use pivot (index=row, columns=col, values=attack rate) to arrange into a grid
# TODO: use sns.heatmap (annot=True) to draw the attack-rate hotspot map
# TODO: identify the region with the highest attack rate, and describe which side the hotspot is concentrated on

## Question 6: Enterovirus Classroom Attack Rate Hotspot Map (Enterovirus Scenario)

An elementary school has an enterovirus cluster; student counts and case counts are available for each grade and class.

1. Compute the attack rate for each grade × classroom
2. Draw a grade × classroom matrix with `pivot` + `sns.heatmap`
3. Compare the overall attack rate across grades, and interpret the difference between lower and upper grades

In [ ]:
# Enterovirus: student counts and cases by grade and classroom at an elementary school (lower grades have higher risk)
rng = np.random.default_rng(863)
_recs = []
for g in range(1, 7):
    for cl in range(1, 6):
        students = int(rng.integers(25, 35))
        risk = max(0.02, 0.28 - g * 0.03)
        cases = rng.binomial(students, risk)
        _recs.append({"grade": g, "classroom": cl, "students": students, "cases": cases})
ev = pd.DataFrame(_recs)
print(f"Enterovirus: {ev['grade'].nunique()} grades × {ev['classroom'].nunique()} classrooms, {ev['cases'].sum()} cases total")

# TODO: compute the attack rate (%) for each grade × classroom = cases / students * 100
# TODO: use pivot (index=grade, columns=classroom) to arrange into a grade × classroom matrix
# TODO: use sns.heatmap to plot the attack rate, and observe which grade is overall higher
# TODO: interpret -- what might the difference in attack rate between lower and upper grades reflect?

## Question 7: Norovirus Banquet Spot Map (Norovirus Scenario)

A norovirus outbreak followed a banquet; seat coordinates, attendee counts, and case counts are available for each table.

1. Compute the attack rate for each table
2. Draw a spot map with `scatter` (point size = attendees, color = attack rate)
3. Mark the tables with high attack rates that cluster together, and infer the likely location of the contamination source

In [ ]:
# Norovirus: seating floor plan for a 25-table banquet (tables near the seafood station have higher attack rates)
rng = np.random.default_rng(874)
_recs = []
for t in range(1, 26):
    x, y = (t - 1) % 5, (t - 1) // 5
    attendees = int(rng.integers(8, 12))
    near_seafood = (x <= 1 and y <= 1)   # bottom-left corner, near the seafood station
    ar = 0.6 if near_seafood else 0.1
    cases = rng.binomial(attendees, ar)
    _recs.append({"table": t, "x": x, "y": y, "attendees": attendees, "cases": cases})
noro = pd.DataFrame(_recs)
print(f"Norovirus banquet: {len(noro)} tables, {noro['attendees'].sum()} attendees, {noro['cases'].sum()} became ill")

# TODO: compute the attack rate for each table = cases / attendees
# TODO: use scatter to plot the banquet floor plan (x, y = seat position, point size = attendees, color = attack rate)
# TODO: mark the "cluster" of tables with a noticeably high attack rate
# TODO: interpret -- which likely contamination source does this spatial cluster point to?

## Question 8 (Challenge): Spatial Analysis of Tuberculosis by Township (Tuberculosis Scenario)

Tuberculosis notifications from 12 townships, with widely varying population sizes.

1. Compute the incidence rate per 100,000 population for each township
2. Compare whether the top-ranked lists by "case count" and by "incidence rate" are the same
3. Explain why the incidence rate is unstable in low-population townships (the small-denominator problem)
4. Compute the correlation between the crowding index and incidence rate
5. Interpret: when mapping townships, should you display case counts or incidence rates? How should low-population areas be handled?

In [ ]:
# Tuberculosis: population, crowding index, and cases for 12 townships (large population differences → unstable small-denominator rates)
rng = np.random.default_rng(885)
_recs = []
for i in range(1, 13):
    popn = int(rng.integers(3000, 120000))
    crowding = round(float(rng.uniform(0.5, 2.0)), 2)
    cases = rng.poisson(15 * crowding * popn / 100000)
    _recs.append({"township": f"T{i:02d}", "population": popn,
                  "crowding_index": crowding, "cases": cases})
tb = pd.DataFrame(_recs)
print(f"Tuberculosis: {len(tb)} townships, population {tb['population'].min():,}–{tb['population'].max():,}")

# TODO: compute the incidence rate per 100,000 for each township = cases / population * 100000
# TODO: sort separately by "case count" and by "incidence rate", and compare whether the two top-ranked lists match
# TODO: explain why the incidence rate for very low-population townships may be unstable (the small-denominator problem)
# TODO: compute the correlation between crowding_index and incidence rate to assess whether crowding is a spatial risk factor
# TODO: interpret -- when making a tuberculosis township map, would you display case counts or incidence rates? Why?